In [ ]:

!pip install vaderSentiment scikit-learn pandas numpy scipy matplotlib seaborn -q
print("✓ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.7 MB/s eta 0:00:00
✓ All packages installed


In [ ]:
import os
import zipfile

file_list = [
    "IRA_tweets_combined_labeled.zip",
    "viral_content.csv",
    "combined_fake_news_dataset.csv",
    "polls_2016.csv"
]

uploaded = {}

for f in file_list:
    if os.path.exists(f):

        if f.endswith('.zip'):
            with zipfile.ZipFile(f, 'r') as zip_ref:

                zip_ref.extractall(".")
                internal_files = zip_ref.namelist()

                csv_name = internal_files[0]
                with open(csv_name, 'rb') as file_data:
                    uploaded[csv_name] = file_data.read()
                print(f"✓ {f} unzipped and {csv_name} loaded into memory.")
        else:
            with open(f, 'rb') as file_data:
                uploaded[f] = file_data.read()
            print(f"✓ {f} loaded from local storage.")
    else:
        print(f"⚠ {f} not found! Please make sure it is uploaded to the folder icon on the left.")

✓ IRA_tweets_combined_labeled.zip unzipped and IRA_tweets_combined_labeled.csv loaded into memory.
✓ viral_content.csv loaded from local storage.
✓ combined_fake_news_dataset.csv loaded from local storage.
✓ polls_2016.csv loaded from local storage.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from matplotlib.lines import Line2D
from collections import Counter
import re, io, os, zipfile, warnings
from google.colab import files as colab_files
warnings.filterwarnings("ignore")

SAVE = dict(bbox_inches="tight", facecolor="white")
STOP = {
    "the","a","an","is","it","in","on","of","to","and","or","but","for",
    "with","this","that","are","was","be","as","at","by","we","he","she",
    "they","our","you","i","my","his","her","its","do","not","from","has",
    "have","had","will","would","could","should","rt","https","http",
    "t.co","amp","via","just","like","get","got","so","if","can","all",
    "more","about","what","your","been","were","s","re","ve","ll","d","m"
}

def find_file(keywords):
    for fname in uploaded.keys():
        if any(k.lower() in fname.lower() for k in keywords):
            return fname
    return None

def read_csv(fname):
    df = pd.read_csv(io.BytesIO(uploaded[fname]), encoding="utf-8", low_memory=False)
    df.columns = df.columns.str.strip().str.lower()
    return df

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def top_words(texts, n=20):
    words = []
    for t in texts.astype(str):
        words += [w for w in re.findall(r"\b[a-z]{3,}\b", t.lower())
                  if w not in STOP]
    return Counter(words).most_common(n)



In [ ]:



print("\n" + "="*60)
print("  STAGE 1: LOADING DATASETS")
print("="*60)

f_ira = find_file(["ira","troll","tweet","2016"])
print(f"\n[1/4] IRA tweets: {f_ira}")
ira = read_csv(f_ira)
ira["publish_date"] = pd.to_datetime(ira["publish_date"], errors="coerce", utc=True)

ira["retweet"]   = pd.to_numeric(pd.Series(ira.get("retweet",   0), index=ira.index), errors="coerce").fillna(0)
ira["followers"] = pd.to_numeric(pd.Series(ira.get("followers", 0), index=ira.index), errors="coerce").fillna(0)
ira["content"]   = ira["content"].astype(str).str.strip()
ira["clean"]     = ira["content"].apply(clean_text)
if "language" in ira.columns:
    ira = ira[ira["language"].str.lower().isin(["english","en"])]
ira = ira[ira["publish_date"].dt.year == 2016].reset_index(drop=True)
ira["date"] = ira["publish_date"].dt.date
ira["week"] = ira["publish_date"].dt.to_period("W").astype(str)
SIDE_MAP = {"RightTroll":"RIGHT","LeftTroll":"LEFT","NewsFeed":"NEUTRAL",
            "HashtagGamer":"BOT","Fearmonger":"RIGHT","Commercial":"NEUTRAL",
            "Unknown":"OTHER","NonEnglish":"OTHER"}
ira["side"] = ira["account_category"].map(SIDE_MAP).fillna("OTHER")
print(f"  → {len(ira):,} tweets | {ira['side'].value_counts().to_dict()}")

# ── Fake news
f_fake = find_file(["fake","news","article"])
print(f"[2/4] Fake news: {f_fake}")
fake = read_csv(f_fake)
fake["is_fake"] = (~fake["id"].astype(str).str.startswith("Real")).astype(int)
fake["clean"]   = (fake["title"].fillna("") + " " + fake["text"].fillna("")).apply(clean_text)
fake["domain"]  = fake["url"].astype(str).str.extract(r"https?://(?:www\.)?([^/]+)")[0].str.lower()
fake_domains    = set(d for d in fake[fake["is_fake"]==1]["domain"].dropna().unique() if len(d) > 4)
print(f"  → {len(fake):,} articles | {fake['is_fake'].sum()} fake | {(fake['is_fake']==0).sum()} real")

# ── Viral content
f_viral = find_file(["viral","social","likes"])
print(f"[3/4] Viral content: {f_viral}")
viral_raw = read_csv(f_viral)
viral_raw["publish_date"]     = pd.to_datetime(viral_raw["date_time"], errors="coerce", utc=True)
viral_raw["date"]             = viral_raw["publish_date"].dt.date
viral_raw["number_of_likes"]  = pd.to_numeric(viral_raw["number_of_likes"],  errors="coerce").fillna(0)
viral_raw["number_of_shares"] = pd.to_numeric(viral_raw["number_of_shares"], errors="coerce").fillna(0)
viral_raw["total_eng"]        = viral_raw["number_of_likes"] + viral_raw["number_of_shares"]
viral_raw["clean"]            = viral_raw["content"].astype(str).apply(clean_text)
viral_raw = viral_raw[viral_raw["publish_date"].dt.year == 2016].reset_index(drop=True)
# label virality: top 25% engagement = viral
v75 = viral_raw["total_eng"].quantile(0.75)
viral_raw["is_viral_label"] = (viral_raw["total_eng"] >= v75).astype(int)
print(f"  → {len(viral_raw):,} posts | {viral_raw['is_viral_label'].sum()} viral")

# ── Polls
f_poll = find_file(["poll","538","forecast","election"])
print(f"[4/4] Polls: {f_poll}")
polls = read_csv(f_poll)
polls["fdate"]           = pd.to_datetime(polls["forecastdate"], errors="coerce", utc=True)
polls["week"]            = polls["fdate"].dt.to_period("W").astype(str)
polls["adjpoll_clinton"] = pd.to_numeric(polls["adjpoll_clinton"], errors="coerce")
polls["adjpoll_trump"]   = pd.to_numeric(polls["adjpoll_trump"],   errors="coerce")
weekly_polls = (polls.groupby("week")
                     .agg(clinton=("adjpoll_clinton","median"),
                          trump  =("adjpoll_trump",  "median"))
                     .reset_index())
weekly_polls["poll_gap"]   = weekly_polls["clinton"] - weekly_polls["trump"]
weekly_polls["poll_swing"] = weekly_polls["poll_gap"].diff().abs().fillna(0)
ira = ira.merge(weekly_polls[["week","poll_gap","poll_swing","clinton","trump"]],
                on="week", how="left")
ira["poll_gap"]   = ira["poll_gap"].fillna(0)
ira["poll_swing"] = ira["poll_swing"].fillna(0)
ira["clinton"]    = ira["clinton"].fillna(ira["clinton"].mean())
ira["trump"]      = ira["trump"].fillna(ira["trump"].mean())

daily_viral = (viral_raw.groupby("date")
                        .agg(med_likes  =("number_of_likes", "median"),
                             med_shares =("number_of_shares","median"))
                        .reset_index())
ira = ira.merge(daily_viral, on="date", how="left")
ira["med_likes"]  = ira["med_likes"].fillna(0)
ira["med_shares"] = ira["med_shares"].fillna(0)
ira["word_count"] = ira["content"].str.split().str.len().fillna(0)

print(f"\n✓ All datasets loaded")


  STAGE 1: LOADING DATASETS

[1/4] IRA tweets: IRA_tweets_combined_labeled.csv
  → 785,267 tweets | {'NEUTRAL': 252528, 'LEFT': 196829, 'BOT': 164315, 'RIGHT': 158923, 'OTHER': 12672}
[2/4] Fake news: combined_fake_news_dataset.csv
  → 21,972 articles | 21770 fake | 202 real
[3/4] Viral content: viral_content.csv
  → 7,551 posts | 1888 viral
[4/4] Polls: polls_2016.csv

✓ All datasets loaded


In [ ]:

print("\n" + "="*60)
print("  STAGE 2: VADER SCORING")
print("="*60)

sid = SentimentIntensityAnalyzer()
print("Running VADER...")
vader = ira["content"].apply(lambda x: sid.polarity_scores(str(x)))
ira["vader_compound"]  = vader.apply(lambda s: s["compound"])
ira["extremity_score"] = ira["vader_compound"].abs()
ira["sentiment_label"] = pd.cut(ira["vader_compound"],
    bins=[-1.01,-0.5,-0.05,0.05,0.5,1.01],
    labels=["very negative","negative","neutral","positive","very positive"])

ira["amp_score"] = np.log1p(ira["retweet"] + ira["med_likes"] + ira["med_shares"])
ira["amp_quartile"] = pd.qcut(
    ira["amp_score"].rank(method="first"), q=4,
    labels=["Q1_low","Q2","Q3","Q4_high"])
ira["is_viral"] = (ira["amp_quartile"] == "Q4_high")

RIGHT_KW = ["crooked","hillary","corrupt","benghazi","emails","drain the swamp",
            "maga","make america great","illegal alien","border wall","deep state",
            "second amendment","soros","globalist","antifa","socialist","communist"]
LEFT_KW  = ["racist","fascist","bigot","sexist","resist","notmypresident",
            "impeach","white supremacist","misogyn","xenophob","islamophob",
            "income inequality","blacklivesmatter","wall street","grab em"]
rp = re.compile("|".join(re.escape(k) for k in RIGHT_KW), re.IGNORECASE)
lp = re.compile("|".join(re.escape(k) for k in LEFT_KW),  re.IGNORECASE)

def get_lean(text):
    t = str(text).lower()
    r = len(rp.findall(t)); l = len(lp.findall(t))
    return (r-l)/(r+l) if (r+l) > 0 else 0.0

ira["partisan_lean"]     = ira["content"].apply(get_lean)
ira["partisan_lean_abs"] = ira["partisan_lean"].abs()

TOXIC = ["hate","kill","destroy","enemy","traitor","liar","scum","idiot","moron",
         "evil","disgusting","pathetic","loser","criminal","thug","terrorist"]
tp = re.compile("|".join(r"\b"+re.escape(w)+r"\b" for w in TOXIC), re.IGNORECASE)
ira["toxicity_score"] = ira["content"].apply(
    lambda x: min(len(tp.findall(str(x)))/max(len(str(x).split()),1)*10, 1.0))

# fake domain flag
ira["references_fakenews"] = ira["content"].apply(
    lambda x: int(any(d in str(x).lower() for d in fake_domains)))
print(f"  → Scoring done | mean extremity={ira['extremity_score'].mean():.3f}")


  STAGE 2: VADER SCORING
Running VADER...
  → Scoring done | mean extremity=0.309


In [ ]:

print("\n" + "="*60)
print("  STAGE 3: ML MODEL A — FAKE NEWS DETECTOR")
print("="*60)

# Train on fake news articles
print("Training fake news classifier on fake news CSV...")
tfidf_fake = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                              min_df=2, strip_accents="unicode")
X_fake_train = tfidf_fake.fit_transform(fake["clean"].fillna(""))
y_fake_train = fake["is_fake"].values

# Check if y_fake_train has at least two classes
if len(np.unique(y_fake_train)) < 2:
    print("  ⚠ Skipping fake news classifier training: Only one class found in training data (all articles are 'real').")

    ira["fake_prob"] = 0.0 # Assign a neutral probability
    ira["predicted_fake"] = 0 # Default all to real
    cv_fake_mean = 0.0
    cv_fake_std = 0.0
    n_fake = 0
    n_real = len(ira)
else:
    clf_fake = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf_fake.fit(X_fake_train, y_fake_train)

    # Cross-validation score on training data
    cv_fake = cross_val_score(clf_fake, X_fake_train, y_fake_train, cv=5, scoring="f1")
    cv_fake_mean = cv_fake.mean()
    cv_fake_std = cv_fake.std()
    print(f"  → Fake news model CV F1: {cv_fake_mean:.3f} \u00b1 {cv_fake_std:.3f}")

    # Predict fake probability on every IRA tweet
    X_ira_fake = tfidf_fake.transform(ira["clean"].fillna(""))
    ira["fake_prob"] = clf_fake.predict_proba(X_ira_fake)[:, 1]
    ira["predicted_fake"] = (ira["fake_prob"] >= 0.5).astype(int)

    n_fake  = ira["predicted_fake"].sum()
    n_real  = (ira["predicted_fake"]==0).sum()

print(f"  \u2192 Fake news model CV F1: {cv_fake_mean:.3f} \u00b1 {cv_fake_std:.3f}") # Moved outside conditional
print(f"  \u2192 IRA tweets predicted FAKE : {n_fake:,} ({n_fake/len(ira)*100:.1f}%)")
print(f"  \u2192 IRA tweets predicted REAL : {n_real:,} ({n_real/len(ira)*100:.1f}%)")


  STAGE 3: ML MODEL A — FAKE NEWS DETECTOR
Training fake news classifier on fake news CSV...
  → Fake news model CV F1: 0.996 ± 0.003
  → Fake news model CV F1: 0.996 ± 0.003
  → IRA tweets predicted FAKE : 785,267 (100.0%)
  → IRA tweets predicted REAL : 0 (0.0%)


In [ ]:


print("\n" + "="*60)
print("  STAGE 4: ML MODEL B — VIRALITY PREDICTOR")
print("="*60)

print("Training virality classifier on viral content CSV...")
tfidf_viral = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                               min_df=2, strip_accents="unicode")
X_viral_train = tfidf_viral.fit_transform(viral_raw["clean"].fillna(""))
y_viral_train = viral_raw["is_viral_label"].values

clf_viral = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf_viral.fit(X_viral_train, y_viral_train)

cv_viral = cross_val_score(clf_viral, X_viral_train, y_viral_train, cv=5, scoring="f1")
print(f"  → Viral model CV F1: {cv_viral.mean():.3f} ± {cv_viral.std():.3f}")

# Predict virality on IRA tweets
X_ira_viral = tfidf_viral.transform(ira["clean"].fillna(""))
ira["viral_prob"] = clf_viral.predict_proba(X_ira_viral)[:, 1]
ira["predicted_viral"] = (ira["viral_prob"] >= 0.5).astype(int)

n_viral_pred = ira["predicted_viral"].sum()
print(f"  → IRA tweets predicted VIRAL     : {n_viral_pred:,} ({n_viral_pred/len(ira)*100:.1f}%)")


  STAGE 4: ML MODEL B — VIRALITY PREDICTOR
Training virality classifier on viral content CSV...
  → Viral model CV F1: 0.169 ± 0.085
  → IRA tweets predicted VIRAL     : 16,752 (2.1%)


In [ ]:


print("\n" + "="*60)
print("  STAGE 5: ML MODEL C — MANIPULATIVE LANGUAGE DETECTOR")
print("="*60)

# Manipulative language = high emotional extremity + high engagement in viral data
viral_raw_vader = viral_raw["clean"].apply(
    lambda x: abs(sid.polarity_scores(str(x))["compound"]))
viral_raw["extremity"] = viral_raw_vader
# label = viral AND emotionally extreme
viral_raw["is_manipulative"] = (
    (viral_raw["is_viral_label"] == 1) &
    (viral_raw["extremity"] >= viral_raw["extremity"].quantile(0.6))
).astype(int)

X_manip = tfidf_viral.transform(viral_raw["clean"].fillna(""))
y_manip = viral_raw["is_manipulative"].values

clf_manip = LogisticRegression(max_iter=1000, C=0.5, random_state=42)
clf_manip.fit(X_manip, y_manip)
cv_manip = cross_val_score(clf_manip, X_manip, y_manip, cv=5, scoring="f1")
print(f"  → Manipulation model CV F1: {cv_manip.mean():.3f} ± {cv_manip.std():.3f}")

# Predict on IRA tweets
ira["manip_prob"] = clf_manip.predict_proba(X_ira_viral)[:, 1]
ira["predicted_manip"] = (ira["manip_prob"] >= 0.5).astype(int)

n_manip = ira["predicted_manip"].sum()
print(f"  → IRA tweets predicted MANIPULATIVE: {n_manip:,} ({n_manip/len(ira)*100:.1f}%)")

# ── Summary table ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  IRA TWEET CLASSIFICATION SUMMARY")
print(f"{'='*60}")
print(f"  Total tweets analyzed   : {len(ira):,}")
print(f"  Predicted FAKE          : {ira['predicted_fake'].sum():,}  ({ira['predicted_fake'].mean()*100:.1f}%)")
print(f"  Predicted REAL          : {(ira['predicted_fake']==0).sum():,}  ({(1-ira['predicted_fake'].mean())*100:.1f}%)")
print(f"  Predicted VIRAL         : {ira['predicted_viral'].sum():,}  ({ira['predicted_viral'].mean()*100:.1f}%)")
print(f"  Predicted MANIPULATIVE  : {ira['predicted_manip'].sum():,}  ({ira['predicted_manip'].mean()*100:.1f}%)")
print(f"  FAKE + VIRAL            : {((ira['predicted_fake']==1)&(ira['predicted_viral']==1)).sum():,}")
print(f"  MANIPULATIVE + VIRAL    : {((ira['predicted_manip']==1)&(ira['predicted_viral']==1)).sum():,}")


  STAGE 5: ML MODEL C — MANIPULATIVE LANGUAGE DETECTOR
  → Manipulation model CV F1: 0.096 ± 0.063
  → IRA tweets predicted MANIPULATIVE: 733 (0.1%)

  IRA TWEET CLASSIFICATION SUMMARY
  Total tweets analyzed   : 785,267
  Predicted FAKE          : 785,267  (100.0%)
  Predicted REAL          : 0  (0.0%)
  Predicted VIRAL         : 16,752  (2.1%)
  Predicted MANIPULATIVE  : 733  (0.1%)
  FAKE + VIRAL            : 16,752
  MANIPULATIVE + VIRAL    : 669


In [ ]:
tweets = ira.copy()
tweets["engagement"] = tweets["retweet"] + tweets["med_likes"] + tweets["med_shares"]
tweets["fake_score"] = tweets["fake_prob"]

# daily aggregation for time-series graphs
daily = (tweets.groupby("date")
               .agg(avg_eng         = ("engagement",      "mean"),
                    avg_fake        = ("predicted_fake",  "mean"),
                    avg_viral       = ("predicted_viral", "mean"),
                    avg_manip       = ("predicted_manip", "mean"),
                    avg_extremity   = ("extremity_score", "mean"),
                    clinton_tweets  = ("content",
                    lambda x: x.str.lower().str.contains("clinton|hillary").sum()),
                    trump_tweets    = ("content",
                    lambda x: x.str.lower().str.contains("trump|donald").sum()),
                    clinton_viral   = ("predicted_viral",
                    lambda x: x[tweets.loc[x.index,"content"].str.lower()
                                      .str.contains("clinton|hillary")].sum()),
                    trump_viral     = ("predicted_viral",
                    lambda x: x[tweets.loc[x.index,"content"].str.lower()
                                      .str.contains("trump|donald")].sum()))
               .reset_index())
daily["date"] = pd.to_datetime(daily["date"])

EVENTS = {
    "2016-09-26":"Debate 1","2016-10-04":"VP Debate",
    "2016-10-07":"Access Hollywood","2016-10-09":"Debate 2",
    "2016-10-19":"Debate 3","2016-10-28":"Comey letter",
    "2016-11-08":"Election Day",
}

# weekly poll + engagement for dual-line
ira["week_start"] = ira["publish_date"].dt.to_period("W").apply(lambda p: p.start_time)
weekly_eng = (ira.groupby("week_start")
                 .agg(avg_eng=("amp_score","mean"),
                      clinton=("clinton","mean"),
                      trump  =("trump",  "mean"))
                 .reset_index())
weekly_eng["week_start"] = pd.to_datetime(weekly_eng["week_start"])

COLORS = {"RIGHT":"#E24B4A","LEFT":"#378ADD","NEUTRAL":"#888780",
          "BOT":"#B4B2A9","OTHER":"#D3D1C7","high":"#D85A30","all":"#5F5E5A"}

plt.rcParams.update({"font.size":11,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":True,
                     "grid.alpha":0.3,"grid.linestyle":"--","figure.dpi":130})
fig_files = []

# ════════════════════════════════════════════════════════════
# GRAPH 1 — Sentiment Distribution
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(tweets["vader_compound"], bins=60, color="#378ADD",
        edgecolor="white", linewidth=0.4, alpha=0.85)
ax.axvline(0, color="#E24B4A", lw=1.5, ls="--", label="Neutral (0)")
ax.axvline(tweets["vader_compound"].mean(), color="#D85A30", lw=2,
           label=f"Mean ({tweets['vader_compound'].mean():.3f})")
ax.set_title("Sentiment Distribution Across Political Tweets", fontsize=13, pad=12)
ax.set_xlabel("VADER Sentiment Score  (−1 very negative → +1 very positive)")
ax.set_ylabel("Number of Tweets")
ax.legend()
plt.tight_layout()
plt.savefig("graph01_sentiment_distribution.png", **SAVE)
fig_files.append("graph01_sentiment_distribution.png")
plt.show(); print("✓ Graph 1")

# ════════════════════════════════════════════════════════════
# GRAPH 2 — Engagement vs Polarization (scatter, ML fake_prob color)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,5))
samp = tweets.sample(min(5000,len(tweets)), random_state=42)
sc = ax.scatter(samp["extremity_score"], samp["engagement"],
                c=samp["fake_prob"], cmap="RdYlGn_r",
                alpha=0.25, s=12, linewidths=0, vmin=0, vmax=1)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("ML Fake Probability")
m2,b2,r2,p2,_ = stats.linregress(samp["extremity_score"], samp["engagement"])
xr = np.linspace(0,1,100)
ax.plot(xr, m2*xr+b2, color="#D85A30", lw=2,
        label=f"Trend  r={r2:.3f}  p={'<0.001' if p2<0.001 else f'{p2:.3f}'}")
ax.set_title("Engagement vs Polarization\n(color = ML fake probability)", fontsize=12, pad=12)
ax.set_xlabel("Polarization Score  |VADER compound|")
ax.set_ylabel("Engagement  (retweets + likes + shares)")
ax.legend()
plt.tight_layout()
plt.savefig("graph02_engagement_vs_polarization.png", **SAVE)
fig_files.append("graph02_engagement_vs_polarization.png")
plt.show(); print("✓ Graph 2")

# ════════════════════════════════════════════════════════════
# GRAPH 3 — Viral vs Non-Viral: Average Engagement
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7,5))
grp3 = tweets.groupby("predicted_viral")["engagement"].agg(["mean","sem"]).reset_index()
grp3["label"] = grp3["predicted_viral"].map({1:"ML-Predicted Viral", 0:"ML-Predicted Non-Viral"})
bars3 = ax.bar(grp3["label"], grp3["mean"],
               color=["#E24B4A","#378ADD"], yerr=grp3["sem"],
               capsize=6, alpha=0.85, width=0.5)
for bar, (_, row) in zip(bars3, grp3.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f"{row['mean']:.1f}", ha="center", va="bottom", fontsize=11)
ax.set_title("Average Engagement: ML-Predicted Viral vs Non-Viral Tweets",
             fontsize=11, pad=12)
ax.set_xlabel("ML Predicted Category")
ax.set_ylabel("Mean Engagement")
plt.tight_layout()
plt.savefig("graph03_viral_vs_nonviral_engagement.png", **SAVE)
fig_files.append("graph03_viral_vs_nonviral_engagement.png")
plt.show(); print("✓ Graph 3")

# ════════════════════════════════════════════════════════════
# GRAPH 4 — Fake Probability vs Engagement (ML output)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,5))
samp4 = tweets.sample(min(5000,len(tweets)), random_state=1)
ax.scatter(samp4["fake_prob"], samp4["engagement"],
           alpha=0.15, s=10, color="#888780", linewidths=0)
bins4 = pd.cut(samp4["fake_prob"], bins=10)
mean4 = samp4.groupby(bins4, observed=True)["engagement"].mean()
centers4 = [iv.mid for iv in mean4.index]
ax.plot(centers4, mean4.values, color="#E24B4A", lw=2.5,
        marker="o", ms=6, label="Mean engagement per bin")
ax.set_title("ML Fake Probability vs Engagement", fontsize=13, pad=12)
ax.set_xlabel("ML Fake Probability  (0 = likely real, 1 = likely fake)")
ax.set_ylabel("Engagement  (retweets + likes + shares)")
ax.legend()
plt.tight_layout()
plt.savefig("graph04_fakescore_vs_engagement.png", **SAVE)
fig_files.append("graph04_fakescore_vs_engagement.png")
plt.show(); print("✓ Graph 4")

# ════════════════════════════════════════════════════════════
# GRAPH 5 — % Fake in Viral vs Non-Viral (ML labels)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8,5))
cats5  = ["ML Non-Viral\n(predicted)","ML Viral\n(predicted)",
          "Truly Fake\n(ML fake prob>0.5)","Fake + Viral\n(both)"]
pcts5  = [
    tweets[tweets["predicted_viral"]==0]["predicted_fake"].mean()*100,
    tweets[tweets["predicted_viral"]==1]["predicted_fake"].mean()*100,
    tweets["predicted_fake"].mean()*100,
    ((tweets["predicted_fake"]==1) & (tweets["predicted_viral"]==1)).mean()*100,
]
clrs5 = ["#378ADD","#E24B4A","#D85A30","#993C1D"]
bars5 = ax.bar(cats5, pcts5, color=clrs5, alpha=0.85, width=0.55)
for bar, pct in zip(bars5, pcts5):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=11)
ax.set_title("Percentage of ML-Predicted Fake Content Across Tweet Categories",
             fontsize=11, pad=12)
ax.set_ylabel("% Tweets Predicted as Fake by ML Model")
plt.tight_layout()
plt.savefig("graph05_fake_pct_viral_nonviral.png", **SAVE)
fig_files.append("graph05_fake_pct_viral_nonviral.png")
plt.show(); print("✓ Graph 5")

# ════════════════════════════════════════════════════════════
# GRAPH 6 — Engagement Distribution Box Plot (ML viral labels)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8,5))
d6_nv = tweets[tweets["predicted_viral"]==0]["engagement"].clip(upper=500).values
d6_v  = tweets[tweets["predicted_viral"]==1]["engagement"].clip(upper=500).values
bp = ax.boxplot([d6_nv, d6_v],
                labels=["ML Non-Viral","ML Viral"],
                patch_artist=True, notch=False,
                medianprops=dict(color="#2C2C2A", linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))
bp["boxes"][0].set_facecolor("#378ADD"); bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor("#E24B4A"); bp["boxes"][1].set_alpha(0.6)
ax.set_title("Engagement Distribution: ML-Predicted Viral vs Non-Viral",
             fontsize=12, pad=12)
ax.set_xlabel("ML Predicted Category")
ax.set_ylabel("Engagement  (clipped at 500)")
plt.tight_layout()
plt.savefig("graph06_engagement_boxplot.png", **SAVE)
fig_files.append("graph06_engagement_boxplot.png")
plt.show(); print("✓ Graph 6")

# ════════════════════════════════════════════════════════════
# GRAPH 7 — Correlation Heatmap
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8,7))
cols7 = ["vader_compound","extremity_score","fake_prob","viral_prob",
         "manip_prob","engagement","toxicity_score","partisan_lean_abs"]
labels7 = ["Sentiment","Polarization","Fake prob\n(ML)","Viral prob\n(ML)",
           "Manip prob\n(ML)","Engagement","Toxicity","Partisan\nlean"]
corr7 = tweets[cols7].rename(columns=dict(zip(cols7,labels7))).corr()
sns.heatmap(corr7, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size":9})
ax.set_title("Correlation Heatmap\n(ML scores + engagement + sentiment)",
             fontsize=12, pad=12)
plt.tight_layout()
plt.savefig("graph07_correlation_heatmap.png", **SAVE)
fig_files.append("graph07_correlation_heatmap.png")
plt.show(); print("✓ Graph 7")

# ════════════════════════════════════════════════════════════
# GRAPH 8 — Top Words in ML-Predicted Viral Tweets
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10,5))
top8 = top_words(tweets[tweets["predicted_viral"]==1]["clean"], n=20)
if top8:
    w8, c8 = zip(*top8)
    ax.barh(list(w8)[::-1], list(c8)[::-1], color="#D85A30", alpha=0.85)
ax.set_title("Top 20 Words in ML-Predicted Viral Tweets", fontsize=13, pad=12)
ax.set_xlabel("Frequency"); ax.set_ylabel("Word")
plt.tight_layout()
plt.savefig("graph08_top_words_viral.png", **SAVE)
fig_files.append("graph08_top_words_viral.png")
plt.show(); print("✓ Graph 8")

# ════════════════════════════════════════════════════════════
# GRAPH 9 — Most Manipulative Words (from ML Model C)
# Top TF-IDF features driving manipulation prediction
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10,5))
feat_names9 = tfidf_viral.get_feature_names_out()
coef9 = clf_manip.coef_[0]
top_manip_idx = np.argsort(coef9)[-20:]
top_manip_words = feat_names9[top_manip_idx]
top_manip_coefs = coef9[top_manip_idx]
ax.barh(top_manip_words, top_manip_coefs, color="#993C1D", alpha=0.85)
ax.set_title("Top 20 Manipulative Language Signals\n(highest coefficients in ML manipulation model)",
             fontsize=11, pad=12)
ax.set_xlabel("Model Coefficient  (higher = stronger manipulation signal)")
ax.set_ylabel("Word / Phrase")
plt.tight_layout()
plt.savefig("graph09_manipulative_words.png", **SAVE)
fig_files.append("graph09_manipulative_words.png")
plt.show(); print("✓ Graph 9")

# ════════════════════════════════════════════════════════════
# GRAPH 10 — Words Driving Virality vs Non-Virality (ML coefs)
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11,6))
feat10  = tfidf_viral.get_feature_names_out()
coef10  = clf_viral.coef_[0]
top_v   = np.argsort(coef10)[-15:]   # most viral
top_nv  = np.argsort(coef10)[:15]    # most non-viral
idx10   = np.concatenate([top_nv, top_v])
words10 = feat10[idx10]
vals10  = coef10[idx10]
clrs10  = ["#E24B4A" if v > 0 else "#378ADD" for v in vals10]
ax.barh(words10, vals10, color=clrs10, alpha=0.85)
ax.axvline(0, color="#2C2C2A", lw=1)
ax.set_title("Words Driving Virality vs Non-Virality\n(ML virality model coefficients — red=viral, blue=non-viral)",
             fontsize=11, pad=12)
ax.set_xlabel("Model Coefficient")
ax.set_ylabel("Word / Phrase")
plt.tight_layout()
plt.savefig("graph10_word_diff_viral_nonviral.png", **SAVE)
fig_files.append("graph10_word_diff_viral_nonviral.png")
plt.show(); print("✓ Graph 10")

# ════════════════════════════════════════════════════════════
# GRAPH 11 — Fake + Viral Tweets: IRA Tweet Classification
# Stacked bar showing fake/real/viral/manipulative breakdown
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10,5))
cats11 = ["Predicted\nFake","Predicted\nReal","Predicted\nViral",
          "Predicted\nManipulative","Fake &\nViral","Manip &\nViral"]
vals11 = [
    ira["predicted_fake"].sum(),
    (ira["predicted_fake"]==0).sum(),
    ira["predicted_viral"].sum(),
    ira["predicted_manip"].sum(),
    ((ira["predicted_fake"]==1) & (ira["predicted_viral"]==1)).sum(),
    ((ira["predicted_manip"]==1) & (ira["predicted_viral"]==1)).sum(),
]
clrs11 = ["#E24B4A","#378ADD","#D85A30","#993C1D","#7F77DD","#534AB7"]
bars11 = ax.bar(cats11, vals11, color=clrs11, alpha=0.85, width=0.6)
for bar, val in zip(bars11, vals11):
    pct = val/len(ira)*100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f"{val:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_title("IRA Tweet Classification: ML Model Results\n"
             "Fake / Real / Viral / Manipulative counts", fontsize=12, pad=12)
ax.set_ylabel("Number of Tweets")
plt.tight_layout()
plt.savefig("graph11_ira_classification_summary.png", **SAVE)
fig_files.append("graph11_ira_classification_summary.png")
plt.show(); print("✓ Graph 11")

# ════════════════════════════════════════════════════════════
# GRAPH 12 — Daily Engagement + Fake/Viral Trend Over Time
# ════════════════════════════════════════════════════════════
fig, ax1 = plt.subplots(figsize=(13,5))
roll12 = daily.set_index("date")["avg_eng"].rolling(7).mean()
ax1.plot(daily["date"], daily["avg_eng"],
         color="#378ADD", lw=1, alpha=0.4, label="Daily mean engagement")
ax1.plot(roll12.index, roll12.values,
         color="#378ADD", lw=2.2, label="7-day rolling engagement")
ax1.set_ylabel("Mean Engagement", color="#378ADD")
ax1.tick_params(axis="y", labelcolor="#378ADD")
ax2 = ax1.twinx()
ax2.plot(daily["date"], daily["avg_fake"].rolling(7).mean(),
         color="#E24B4A", lw=1.8, ls="--", label="7-day rolling % fake")
ax2.plot(daily["date"], daily["avg_viral"].rolling(7).mean(),
         color="#D85A30", lw=1.8, ls=":", label="7-day rolling % viral")
ax2.set_ylabel("ML-Predicted Rate (7-day avg)", color="#E24B4A")
ax2.tick_params(axis="y", labelcolor="#E24B4A")
for ds, name in EVENTS.items():
    evt = pd.to_datetime(ds)
    if daily["date"].min() <= evt <= daily["date"].max():
        ax1.axvline(evt, color="#D3D1C7", lw=1, ls=":")
        ax1.text(evt, ax1.get_ylim()[1]*0.95, name,
                 rotation=90, va="top", fontsize=7.5, color="#5F5E5A")
lines1,lab1 = ax1.get_legend_handles_labels()
lines2,lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, loc="upper left", fontsize=8)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
ax1.set_xlabel("Date")
plt.title("Daily Engagement + ML Fake/Viral Rate Over Time (2016)", fontsize=12, pad=12)
plt.tight_layout()
plt.savefig("graph12_daily_trend.png", **SAVE)
fig_files.append("graph12_daily_trend.png")
plt.show(); print("✓ Graph 12")

# ════════════════════════════════════════════════════════════
# GRAPH 13 — Poll Trends vs Viral Influence per Candidate
# KEY GRAPH: Clinton vs Trump polls vs viral tweets about them
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 1, figsize=(13,9), sharex=True)

# ── Top panel: poll numbers ──────────────────────────────────
ax = axes[0]
ax.plot(weekly_eng["week_start"], weekly_eng["clinton"],
        color="#378ADD", lw=2.5, label="Clinton poll % (adj)")
ax.plot(weekly_eng["week_start"], weekly_eng["trump"],
        color="#E24B4A", lw=2.5, label="Trump poll % (adj)")
ax.fill_between(weekly_eng["week_start"],
                weekly_eng["clinton"], weekly_eng["trump"],
                where=weekly_eng["clinton"] >= weekly_eng["trump"],
                alpha=0.1, color="#378ADD", label="Clinton lead")
ax.fill_between(weekly_eng["week_start"],
                weekly_eng["clinton"], weekly_eng["trump"],
                where=weekly_eng["clinton"] < weekly_eng["trump"],
                alpha=0.1, color="#E24B4A", label="Trump lead")
for ds, name in EVENTS.items():
    evt = pd.to_datetime(ds)
    if weekly_eng["week_start"].min() <= evt <= weekly_eng["week_start"].max():
        ax.axvline(evt, color="#D3D1C7", lw=1, ls=":")
        ax.text(evt, ax.get_ylim()[1]*0.97, name,
                rotation=90, va="top", fontsize=7.5, color="#5F5E5A")
ax.set_ylabel("Adjusted Poll %")
ax.set_title("Poll Trends vs Viral Tweet Activity per Candidate (2016)",
             fontsize=12, pad=12)
ax.legend(fontsize=9)

# ── Bottom panel: viral tweet volume per candidate ───────────
ax2 = axes[1]
daily2 = daily.copy()
roll_cl = daily2.set_index("date")["clinton_viral"].rolling(7).mean()
roll_tr = daily2.set_index("date")["trump_viral"].rolling(7).mean()
ax2.plot(roll_cl.index, roll_cl.values,
         color="#378ADD", lw=2, label="Viral tweets about Clinton (7-day avg)")
ax2.plot(roll_tr.index, roll_tr.values,
         color="#E24B4A", lw=2, label="Viral tweets about Trump (7-day avg)")
for ds, name in EVENTS.items():
    evt = pd.to_datetime(ds)
    if daily2["date"].min() <= evt <= daily2["date"].max():
        ax2.axvline(evt, color="#D3D1C7", lw=1, ls=":")
ax2.set_ylabel("# ML-Predicted Viral Tweets (7-day avg)")
ax2.set_xlabel("Date (2016)")
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("graph13_polls_vs_viral_candidates.png", **SAVE)
fig_files.append("graph13_polls_vs_viral_candidates.png")
plt.show(); print("✓ Graph 13")

# ════════════════════════════════════════════════════════════
# GRAPH 14 — Viral Content CSV: Engagement by Is_Viral
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,5))
labs14  = ["Non-Viral\n(bottom 75%)","Viral\n(top 25%)"]
likes14 = [viral_raw[viral_raw["is_viral_label"]==0]["number_of_likes"].mean(),
           viral_raw[viral_raw["is_viral_label"]==1]["number_of_likes"].mean()]
shar14  = [viral_raw[viral_raw["is_viral_label"]==0]["number_of_shares"].mean(),
           viral_raw[viral_raw["is_viral_label"]==1]["number_of_shares"].mean()]
x14 = np.arange(2); w14 = 0.35
ax.bar(x14-w14/2, likes14, width=w14, color="#378ADD", alpha=0.85, label="Mean likes")
ax.bar(x14+w14/2, shar14,  width=w14, color="#E24B4A", alpha=0.85, label="Mean shares")
for i, (l, s) in enumerate(zip(likes14, shar14)):
    ax.text(i-w14/2, l+0.5, f"{l:.0f}", ha="center", fontsize=9)
    ax.text(i+w14/2, s+0.5, f"{s:.0f}", ha="center", fontsize=9)
ax.set_xticks(x14); ax.set_xticklabels(labs14)
ax.set_title("Viral Content Dataset: Likes & Shares by Virality Category",
             fontsize=12, pad=12)
ax.set_xlabel("Viral Content Category")
ax.set_ylabel("Mean Count")
ax.legend()
plt.tight_layout()
plt.savefig("graph14_viral_content_comparison.png", **SAVE)
fig_files.append("graph14_viral_content_comparison.png")
plt.show(); print("✓ Graph 14")

# ════════════════════════════════════════════════════════════
# GRAPH 15 — Feature Importance for ML Virality Prediction
# ════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9,5))
feat_cols15 = ["extremity_score","toxicity_score","fake_prob",
               "manip_prob","partisan_lean_abs","amp_score",
               "word_count","followers"]
df15 = tweets[feat_cols15 + ["predicted_viral"]].dropna()
rf15 = RandomForestClassifier(n_estimators=200, max_depth=8,
                               min_samples_leaf=20, random_state=42, n_jobs=-1)
rf15.fit(df15[feat_cols15], df15["predicted_viral"])
imp15 = pd.DataFrame({"Feature": feat_cols15,
                       "Importance": rf15.feature_importances_}
                     ).sort_values("Importance", ascending=True)
clrs15 = ["#D85A30" if any(k in f for k in ["extremity","toxicity","manip"])
          else "#E24B4A" if "fake" in f
          else "#534AB7" if "partisan" in f
          else "#378ADD" for f in imp15["Feature"]]
ax.barh(imp15["Feature"], imp15["Importance"], color=clrs15, alpha=0.85)
ax.set_title("Feature Importance for ML Virality Prediction\n"
             "(Random Forest trained on IRA tweet features)",
             fontsize=11, pad=12)
ax.set_xlabel("Feature Importance (Gini impurity)")
ax.set_ylabel("Feature")
ax.legend(handles=[
    Line2D([0],[0],color="#D85A30",lw=6,label="Emotional/manipulation"),
    Line2D([0],[0],color="#E24B4A",lw=6,label="Fake news probability"),
    Line2D([0],[0],color="#534AB7",lw=6,label="Partisan lean"),
    Line2D([0],[0],color="#378ADD",lw=6,label="Structural"),
], fontsize=9)
plt.tight_layout()
plt.savefig("graph15_feature_importance.png", **SAVE)
fig_files.append("graph15_feature_importance.png")
plt.show(); print("✓ Graph 15")

# ════════════════════════════════════════════════════════════
# FINAL SUMMARY PRINT
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("  COMPLETE ANALYSIS SUMMARY")
print("="*60)
print(f"\n  IRA Tweets analyzed      : {len(ira):,}")
print(f"\n  ML MODEL PERFORMANCE")
print(f"  Fake news model  CV F1   : {cv_fake_mean:.3f}")
print(f"  Virality model   CV F1   : {cv_viral.mean():.3f}")
print(f"  Manipulation model CV F1 : {cv_manip.mean():.3f}")
print(f"\n  IRA TWEET PREDICTIONS")
print(f"  Predicted FAKE           : {ira['predicted_fake'].sum():,}  ({ira['predicted_fake'].mean()*100:.1f}%)")
print(f"  Predicted REAL           : {(ira['predicted_fake']==0).sum():,}  ({(1-ira['predicted_fake'].mean())*100:.1f}%)")
print(f"  Predicted VIRAL          : {ira['predicted_viral'].sum():,}  ({ira['predicted_viral'].mean()*100:.1f}%)")
print(f"  Predicted MANIPULATIVE   : {ira['predicted_manip'].sum():,}  ({ira['predicted_manip'].mean()*100:.1f}%)")
print(f"  FAKE + VIRAL combined    : {((ira['predicted_fake']==1)&(ira['predicted_viral']==1)).sum():,}")
print(f"  MANIP + VIRAL combined   : {((ira['predicted_manip']==1)&(ira['predicted_viral']==1)).sum():,}")

# per candidate
clinton_tweets = ira[ira["content"].str.lower().str.contains("clinton|hillary", na=False)]
trump_tweets   = ira[ira["content"].str.lower().str.contains("trump|donald",    na=False)]
print(f"\n  CANDIDATE TWEET ANALYSIS")
print(f"  Clinton-related tweets   : {len(clinton_tweets):,}")
print(f"    → Predicted viral      : {clinton_tweets['predicted_viral'].sum():,}  ({clinton_tweets['predicted_viral'].mean()*100:.1f}%)")
print(f"    → Predicted fake       : {clinton_tweets['predicted_fake'].sum():,}  ({clinton_tweets['predicted_fake'].mean()*100:.1f}%)")
print(f"    → Predicted manipul.   : {clinton_tweets['predicted_manip'].sum():,}  ({clinton_tweets['predicted_manip'].mean()*100:.1f}%)")
print(f"  Trump-related tweets     : {len(trump_tweets):,}")
print(f"    → Predicted viral      : {trump_tweets['predicted_viral'].sum():,}  ({trump_tweets['predicted_viral'].mean()*100:.1f}%)")
print(f"    → Predicted fake       : {trump_tweets['predicted_fake'].sum():,}  ({trump_tweets['predicted_fake'].mean()*100:.1f}%)")
print(f"    → Predicted manipul.   : {trump_tweets['predicted_manip'].sum():,}  ({trump_tweets['predicted_manip'].mean()*100:.1f}%)")

✓ Graph 1
✓ Graph 2
✓ Graph 3
✓ Graph 4
✓ Graph 5
✓ Graph 6
✓ Graph 7
✓ Graph 8
✓ Graph 9
✓ Graph 10
✓ Graph 11
✓ Graph 12
✓ Graph 13
✓ Graph 14
✓ Graph 15

  COMPLETE ANALYSIS SUMMARY

  IRA Tweets analyzed      : 785,267

  ML MODEL PERFORMANCE
  Fake news model  CV F1   : 0.996
  Virality model   CV F1   : 0.169
  Manipulation model CV F1 : 0.096

  IRA TWEET PREDICTIONS
  Predicted FAKE           : 785,267  (100.0%)
  Predicted REAL           : 0  (0.0%)
  Predicted VIRAL          : 16,752  (2.1%)
  Predicted MANIPULATIVE   : 733  (0.1%)
  FAKE + VIRAL combined    : 16,752
  MANIP + VIRAL combined   : 669

  CANDIDATE TWEET ANALYSIS
  Clinton-related tweets   : 39,723
    → Predicted viral      : 439  (1.1%)
    → Predicted fake       : 39,723  (100.0%)
    → Predicted manipul.   : 4  (0.0%)
  Trump-related tweets     : 61,467
    → Predicted viral      : 521  (0.8%)
    → Predicted fake       : 61,467  (100.0%)
    → Predicted manipul.   : 23  (0.0%)


In [ ]:
# ════════════════════════════════════════════════════════════
# DOWNLOAD ALL 15 GRAPHS
# ════════════════════════════════════════════════════════════
zip_name = "all_15_graphs_final.zip"
with zipfile.ZipFile(zip_name, "w") as zf:
    for f in fig_files:
        if os.path.exists(f):
            zf.write(f)
print(f"\n  {len(fig_files)} graphs saved:")
for f in fig_files:
    print(f"    {f}")
print(f"\nDownloading {zip_name}...")
colab_files.download(zip_name)
print("✓ Done!")